In [1]:
from langchain_openai import ChatOpenAI
import os 
from dotenv import load_dotenv
load_dotenv()
llm = ChatOpenAI(model="gpt-5-mini")

## Tools

In [2]:
from langchain.tools import tool

In [3]:
@tool
def tool_duckduckgo_search(query: str) -> str:
    
    """Use this tool when you need to answer questions about current events or general knowledge. """

    from langchain_community.tools import DuckDuckGoSearchRun

    search = DuckDuckGoSearchRun()

    response = search.invoke(query)

    return response


In [4]:
tool_duckduckgo_search.invoke("What is the capital of France?")

"With 200,000 inhabitants in 1328, Paris, then alreadythecapitalofFrance,wasthemost populous city of Europe. By comparison, London in 1300 had 80,000 inhabitants.[29] By the early fourteenth century... CapitalofFrance. ParisisthecapitalofFrance, and thereforeistheseatofFrance's national government. For the executive, the two chief officers each have their own official residences, which also serve as their offices. The Panthéon honors someofFrance’s greatest figures, while the Cluny baths and the ancient arenas of Lutetia reveal traces of the Roman past. Walking along Rue Mouffetard, the medieval character ofthecapitalbecomes visible in its charming façades and lively atmosphere. thecapitaland largest cityofFrance; and international center of culture and commerce. Franceis a country in Europe, known for the Eiffel Tower and wine regions. It has a population of 66.7 million, making it the 23rd largest country in the world. Itscapitalis Paris.Francehas a diverse economy with strong touris

In [5]:
@tool 
def tool_wikipedia_search(query: str) -> str:
    
    """Use this tool when you need to answer questions about persons, places, etc. """

    from langchain_community.tools import WikipediaQueryRun
    from langchain_community.utilities import WikipediaAPIWrapper

    wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

    response = wikipedia.invoke(query)

    return response


In [6]:
tool_wikipedia_search.invoke("Sam Altman")

'Page: Sam Altman\nSummary: Samuel Harris Altman (born April 22, 1985) is an American businessman and entrepreneur who has been the chief executive officer (CEO) of the artificial intelligence research organization OpenAI since 2019.\nAltman attended Stanford University for two years before dropping out and co-founding Loopt, a smartphone geosocial networking service, which raised more than US$30 million in venture capital before being acquired by Green Dot Corporation for $43.4 million in cash. In 2011, Altman joined Y Combinator, a technology startup accelerator and venture capital firm, and was the company\'s president from 2014 to 2019. \nAfter co-founding OpenAI in 2015, Altman later became the organization\'s CEO in 2019. In 2023, he was ousted by the organization\'s board of directors, who cited a lack of "confidence in his ability to continue leading OpenAI" in an official post. However, the move was immediately met with significant backlash from employees and investors, result

In [7]:
@tool
def tool_arxiv_search(query: str) -> str:
    
    """Use this tool when you need to answer questions about scientific papers or research topics. """

    from langchain_community.tools import ArxivQueryRun
    from langchain_community.utilities import ArxivAPIWrapper

    # 1. Initialize the arXiv API wrapper
    arxiv_wrapper = ArxivAPIWrapper(
        top_k_results=3,       # Number of papers to retrieve
        doc_content_chars_max=2000  # Max characters per document
    )

    # 2. Create the arXiv tool
    arxiv_tool = ArxivQueryRun(api_wrapper=arxiv_wrapper)

    # 3. Use the tool directly
    result = arxiv_tool.run(query)

    print(result)



In [8]:
@tool
def tool_personal_info(name: str) -> str:
    """Use this tool when you need to answer questions about personal information.
    Args:
        name (str): The name of the person to look up.
    Returns:
        str: A string containing the person's age and occupation, or a message if the information is not found.
    """
    
    infos = [{
        "name": "John Doe",
        "age": 30,
        "occupation": "Software Engineer"
    },
    {
        "name": "Jane Smith",
        "age": 25,
        "occupation": "Data Scientist"
    }]

    for info in infos:
        if info["name"].lower() == name.lower():
            return f"{info['name']} is {info['age']} years old and works as a {info['occupation']}."
    return "Information not found."


In [9]:
tool_personal_info.invoke("John Doe")

'John Doe is 30 years old and works as a Software Engineer.'

In [10]:
@tool
def tool_rag(query: str) -> str:
    """Use this tool when you need to answer questions based on NovaSpehere Organization's documentation.""" 

    from langchain_community.vectorstores import Chroma
    from langchain_openai import OpenAIEmbeddings
    embed_model = OpenAIEmbeddings(model="text-embedding-3-small")
    chroma_db_con = Chroma(persist_directory="./chroma_db_semantic", embedding_function=embed_model)

    # Retrieve relevant documents from the vector store
    relevant_docs = chroma_db_con.similarity_search(query, k=2)
    relevant_docs_content = "\n".join([doc.page_content for doc in relevant_docs])
    return relevant_docs_content
    

In [11]:
tool_rag.invoke("When was NovaSphere Organization founded?")

C:\Users\anshl\AppData\Local\Temp\ipykernel_18572\3480324300.py:8: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  chroma_db_con = Chroma(persist_directory="./chroma_db_semantic", embedding_function=embed_model)


'NovaSphere Technologies is a fictional organization created to represent a modern data \nand technology company that has grown gradually over the years. The organization was \nfounded in 2016 by a small group of software engineers who strongly believed that data \nwould become one of the most valuable assets for every business in the future.\nThe founders also started planning \nthe next stage of growth, which included expanding into new regions and working with \ninternational clients. Today, NovaSphere Technologies is considered a reliable organization that provides data \nengineering and analytics services to companies from different industries such as finance, \nhealthcare, retail, and e-commerce. Even though the company has grown significantly \nsince 2016, the original vision has not changed. The focus is still on learning continuously, \nimproving the quality of work, and helping organizations make better decisions using data. The company believes that the future of business wi

## Bind Tools

In [12]:
toolkit = [
    tool_duckduckgo_search,
    tool_wikipedia_search,
    tool_arxiv_search,
    tool_personal_info,
    tool_rag
]

In [13]:
llm_bind = llm.bind_tools(toolkit)

In [14]:
llm_bind.invoke("What's the age of John Doe?. Make tool calls if necessary.")

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 315, 'total_tokens': 341, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DQFFC4RmjgsY4HEUOj2NcMMjEdUx2', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d4f09-398f-7200-9447-f17cd3c4f215-0', tool_calls=[{'name': 'tool_personal_info', 'args': {'name': 'John Doe'}, 'id': 'call_Cl6zmJkKepV1ooqqpnogjU2L', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 315, 'output_tokens': 26, 'total_tokens': 341, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

## ReAct Agent

In [15]:
from langchain.agents import create_agent


my_agent = create_agent(llm_bind, toolkit)

In [16]:
my_agent.invoke(
    {"messages": [{"role": "user", "content": "What's the age of John Doe?. Make tool calls if necessary."}]}
)

{'messages': [HumanMessage(content="What's the age of John Doe?. Make tool calls if necessary.", additional_kwargs={}, response_metadata={}, id='20ec7736-864f-447a-afed-cfa996394a7f'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 315, 'total_tokens': 341, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DQFFDbG4l4GKlhBQFsofvItbm74yr', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d4f09-400e-7960-9f69-21967c6e5a36-0', tool_calls=[{'name': 'tool_personal_info', 'args': {'name': 'John Doe'}, 'id': 'call_moxIVU0p6tmhi87k6w22h8Mf', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata

In [17]:
my_agent.invoke(
    {"messages": [{"role": "user", "content": "When was NovaSphere Organization founded?"}]}
)

{'messages': [HumanMessage(content='When was NovaSphere Organization founded?', additional_kwargs={}, response_metadata={}, id='dadc60a6-cba4-475f-9989-77e449e0e347'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 309, 'total_tokens': 339, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DQFFHdCvmkOmCT5XvQyUYgztK639h', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d4f09-506e-7192-94ce-8169812c9656-0', tool_calls=[{'name': 'tool_rag', 'args': {'query': 'When was NovaSphere Organization founded?'}, 'id': 'call_CownmN1a5SHashjuM5xUNsTK', 'type': 'tool_call'}], invalid_tool_calls=[], usage_m